# PPO car trainer (GPU)
Set `MAP_PATH` in the first cell (upload the map `.json` next to this notebook), then **Run All**. Prints progress every few updates; checkpoints go to `runs/` (`latest.pt` resume, `best.pt/.npz`, `final.pt/.npz`). The `.npz` files load directly in the local `test_model.py`.

In [ ]:
import json, math, os, time, collections
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical

# ════════════════════════════ CONFIG ════════════════════════════
MAP_PATH     = "circle.json"   # map file saved by environment.py
OUT_DIR      = "runs"          # checkpoints are written here
RESUME       = None            # e.g. "runs/latest.pt" to continue a previous run
SEED         = 0

NUM_ENVS     = 128             # cars simulated in parallel on the GPU
ROLLOUT      = 128             # decisions per car between PPO updates (batch = 16384)
TOTAL_STEPS  = 300_000_000     # stop after this many decisions (summed over all cars)
MAX_HOURS    = None            # wall-clock budget, e.g. 11.5 -> saves and exits cleanly
LOG_EVERY    = 10              # updates between printed log lines
SAVE_EVERY   = 25              # updates between overwriting runs/latest.pt
SNAP_EVERY   = 500             # updates between permanent runs/snap_XXXXX.pt copies
SAFETY_OVERRIDE = False        # hard-coded counter-steer near walls. KEEP OFF: tests showed it makes even a
                               # perfect scripted driver crash 100% at >=160 px/s (it fights the lane-keeping).

# PPO
GAMMA, LAM, CLIP_EPS = 0.995, 0.95, 0.2
LR_ACTOR, LR_CRITIC  = 3e-4, 1e-3
ENT_COEF, ENT_DECAY, ENT_MIN = 0.05, 0.998, 0.01   # entropy bonus, decayed per update
EPOCHS, MINIBATCH    = 4, 2048
TARGET_KL, MAX_GRAD  = 0.03, 0.5
HIDDEN               = 128

# Simulation (identical to map_reader.py, AI mode)
FPS, SUBSTEPS      = 60, 6            # one decision per frame, physics at 360 Hz
FRAME_DT           = 1.0 / FPS
SUB_DT             = FRAME_DT / SUBSTEPS
WHEELBASE          = 28.0
MAX_SPEED          = 240.0
ACCEL, BRAKE_F     = 280.0, 280.0
DRAG_K             = 1.6
MAX_STEER_DEG      = 28.0
MAX_STEER          = math.radians(MAX_STEER_DEG)
STEER_RATE, STEER_RTN = math.radians(90), math.radians(150)
RAY_MAX, RAY_N     = 220.0, 10        # RAY_N rays in the front arc + RAY_N in the rear arc
RAY_FOV            = 160.0
MARGIN             = 6.0              # closer than this to a road edge = crash
STALL_TIMEOUT, STALL_EPS = 5.0, 4.0   # < STALL_EPS px of progress for 5 s = crash
WALL_CRITICAL      = 0.12
CRASH_PENALTY      = -15.0
MAX_EP_STEPS       = 1800             # 30 s episode cap (treated as truncation, not failure)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OBS_DIM, ACT_DIM = 2 * RAY_N + 2, 9
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "(no GPU found!)")


In [ ]:
# ════════════════════ MAP + BATCHED SIMULATOR ════════════════════
with open(MAP_PATH) as f:
    _m = json.load(f)
_t = lambda k: torch.tensor(_m[k], dtype=torch.float32, device=DEVICE)
CENTER, LEFT, RIGHT = _t("center"), _t("left"), _t("right")
WALL_A = torch.cat([LEFT[:-1], RIGHT[:-1]])      # wall segments A -> B
WALL_B = torch.cat([LEFT[1:],  RIGHT[1:]])
WALL_E = WALL_B - WALL_A
WALL_L2 = (WALL_E ** 2).sum(-1).clamp_min(1e-9)
print(f"map {MAP_PATH}: {len(CENTER)} centre pts, {len(WALL_A)} wall segments")

# ray angles relative to heading: RAY_N front rays then RAY_N rear rays
_front = [math.radians(-RAY_FOV / 2 + i * RAY_FOV / (RAY_N - 1)) for i in range(RAY_N)]
_rear  = [math.radians(180 - RAY_FOV / 2 + i * RAY_FOV / (RAY_N - 1)) for i in range(RAY_N)]
REL_ANG = torch.tensor(_front + _rear, dtype=torch.float32, device=DEVICE)

# centre line segments: used ONLY to compute the training reward (the policy never sees them)
CENTER_A, CENTER_B = CENTER[:-1], CENTER[1:]
CENTER_E = CENTER_B - CENTER_A
CENTER_L = CENTER_E.norm(dim=-1).clamp_min(1e-6)
CENTER_T = CENTER_E / CENTER_L[:, None]
HALF_W = float(_m.get("lane_width", 60)) / 2

# action table: COAST THROTTLE BRAKE LEFT RIGHT THR_L THR_R BRK_L BRK_R
ACT_THR = torch.tensor([0, 1, 0, .3, .3, .9, .9, 0, 0], dtype=torch.float32, device=DEVICE)
ACT_BRK = torch.tensor([0, 0, 1, 0, 0, 0, 0, 1, 1], dtype=torch.bool, device=DEVICE)
ACT_STR = torch.tensor([0, 0, 0, -1, 1, -1, 1, -1, 1], dtype=torch.float32, device=DEVICE)


def cast_rays(pos, heading):
    """pos (E,2), heading (E,) -> frac (E, 2*RAY_N): hit distance / RAY_MAX."""
    ang = heading[:, None] + REL_ANG[None, :]                       # (E,R)
    dx, dy = torch.cos(ang)[..., None], torch.sin(ang)[..., None]   # (E,R,1)
    aox = WALL_A[:, 0][None, None, :] - pos[:, 0][:, None, None]    # (E,1,S)
    aoy = WALL_A[:, 1][None, None, :] - pos[:, 1][:, None, None]
    ex, ey = WALL_E[:, 0], WALL_E[:, 1]                             # (S,)
    denom = dx * ey - dy * ex                                       # (E,R,S)
    ok = denom.abs() > 1e-10
    denom = torch.where(ok, denom, torch.ones_like(denom))
    t = (aox * ey - aoy * ex) / denom
    u = (aox * dy - aoy * dx) / denom
    hit = ok & (t > 0) & (u >= 0) & (u <= 1)
    t = torch.where(hit, t, torch.full_like(t, RAY_MAX))
    return t.amin(-1).clamp(max=RAY_MAX) / RAY_MAX


def wall_dist(pos):
    """Distance from each car to the nearest road edge. (E,)"""
    ap = pos[:, None, :] - WALL_A[None]                             # (E,S,2)
    s = ((ap * WALL_E[None]).sum(-1) / WALL_L2).clamp(0, 1)
    closest = WALL_A[None] + s[..., None] * WALL_E[None]
    return (pos[:, None, :] - closest).norm(dim=-1).amin(-1)


def track_state(pos, heading):
    """Distance from the centre line and cos(angle between heading and road direction)."""
    ap = pos[:, None, :] - CENTER_A[None]                              # (E,C-1,2)
    s = ((ap * CENTER_E[None]).sum(-1) / CENTER_L ** 2).clamp(0, 1)
    d = (pos[:, None, :] - (CENTER_A[None] + s[..., None] * CENTER_E[None])).norm(dim=-1)
    lat, j = d.min(1)
    tang = CENTER_T[j]
    return lat, torch.cos(heading) * tang[:, 0] + torch.sin(heading) * tang[:, 1]


def reward_fn(speed, lat, fwd):
    """Progress along the road (speed x alignment with the road direction) minus an off-centre penalty."""
    return 3.0 * speed * fwd / MAX_SPEED - 2.0 * (lat / HALF_W) ** 2


class Cars:
    """NUM_ENVS independent cars on the same map, all state on DEVICE."""
    def __init__(self, n):
        self.n = n
        z = lambda *s: torch.zeros(*s, device=DEVICE)
        self.pos, self.anchor = z(n, 2), z(n, 2)
        self.heading, self.speed, self.steer, self.stall = z(n), z(n), z(n), z(n)
        self.steps = torch.zeros(n, dtype=torch.long, device=DEVICE)
        self.reset(torch.ones(n, dtype=torch.bool, device=DEVICE))

    def reset(self, mask):
        k = int(mask.sum())
        if k == 0:
            return
        i = torch.randint(0, len(CENTER) - 1, (k,), device=DEVICE)   # random spot on the road
        a, b = CENTER[i], CENTER[i + 1]
        self.pos[mask] = a + torch.rand(k, 1, device=DEVICE) * (b - a)
        self.heading[mask] = torch.atan2(b[:, 1] - a[:, 1], b[:, 0] - a[:, 0])
        self.speed[mask] = 0.25 * MAX_SPEED
        self.steer[mask] = 0.0
        self.anchor[mask] = self.pos[mask]
        self.stall[mask] = 0.0
        self.steps[mask] = 0

    def observe(self):
        frac = cast_rays(self.pos, self.heading)
        sn = (self.speed / MAX_SPEED).clamp(-1, 1)
        an = (torch.rad2deg(self.steer) / MAX_STEER_DEG).clamp(-1, 1)
        return torch.cat([frac, sn[:, None], an[:, None]], 1), frac

    def _physics(self, thr, brk, sdir, dt):
        sf = (self.speed.abs() / 20.0).clamp(max=1)
        diff = sdir * MAX_STEER - self.steer
        rate = torch.where(sdir.abs() > 0.01, STEER_RATE * sf, STEER_RTN * sf.clamp(min=0.1))
        self.steer = self.steer + torch.sign(diff) * torch.minimum(diff.abs(), rate * dt)

        v = self.speed
        braked = torch.where(v > 0, (v - BRAKE_F * dt).clamp(min=0),
                             torch.where(v < 0, (v + BRAKE_F * dt).clamp(max=0), v))
        a = v + thr * ACCEL * dt
        drag = (DRAG_K * a * a / MAX_SPEED + 15.0) * dt
        free = torch.where(a > 0, (a - drag).clamp(min=0),
                           torch.where(a < 0, (a + drag).clamp(max=0), a))
        self.speed = torch.where(brk, braked, free).clamp(-0.35 * MAX_SPEED, MAX_SPEED)

        turning = (self.speed.abs() > 0.1) & (self.steer.abs() > 0.0005)
        omega = self.speed * torch.tan(self.steer) / WHEELBASE
        self.heading = self.heading + torch.where(turning, omega * dt, torch.zeros_like(omega))
        self.pos = self.pos + self.speed[:, None] * dt * torch.stack(
            [torch.cos(self.heading), torch.sin(self.heading)], 1)

    def step(self, action):
        """Hold `action` for one frame. Returns obs_next, obs_final, reward, crashed, truncated.
        obs_final is the observation before auto-reset (for truncation bootstrapping)."""
        thr, brk, sdir = ACT_THR[action], ACT_BRK[action], ACT_STR[action]
        crashed = torch.zeros(self.n, dtype=torch.bool, device=DEVICE)
        for _ in range(SUBSTEPS):
            eff = sdir
            if SAFETY_OVERRIDE:
                fr = cast_rays(self.pos, self.heading)[:, :RAY_N]
                side = torch.where(sdir < 0, fr[:, :RAY_N // 2].amin(1), fr[:, RAY_N // 2:].amin(1))
                eff = torch.where((sdir != 0) & (side < WALL_CRITICAL), -sdir, sdir)
            self._physics(thr, brk, eff, SUB_DT)
            crashed |= wall_dist(self.pos) < MARGIN

            far = (self.pos - self.anchor).norm(dim=1) >= STALL_EPS
            self.anchor = torch.where(far[:, None], self.pos, self.anchor)
            self.stall = torch.where(far, torch.zeros_like(self.stall), self.stall + SUB_DT)
            stalled = self.stall >= STALL_TIMEOUT
            crashed |= stalled
            self.stall = torch.where(stalled, torch.zeros_like(self.stall), self.stall)

        self.steps += 1
        obs_final, _ = self.observe()
        lat, fwd = track_state(self.pos, self.heading)
        reward = torch.where(crashed, torch.full_like(self.speed, CRASH_PENALTY),
                             reward_fn(self.speed, lat, fwd))
        trunc = (self.steps >= MAX_EP_STEPS) & ~crashed
        done = crashed | trunc
        if done.any():
            self.reset(done)
            obs_next, _ = self.observe()
        else:
            obs_next = obs_final
        return obs_next, obs_final, reward, crashed, trunc


In [ ]:
# ═══════════════════════ PPO (PyTorch) ═══════════════════════
class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        def mlp(out, gain):
            net = nn.Sequential(nn.Linear(OBS_DIM, HIDDEN), nn.Tanh(),
                                nn.Linear(HIDDEN, HIDDEN), nn.Tanh(), nn.Linear(HIDDEN, out))
            for i in (0, 2):
                nn.init.orthogonal_(net[i].weight, math.sqrt(2)); nn.init.zeros_(net[i].bias)
            nn.init.orthogonal_(net[4].weight, gain); nn.init.zeros_(net[4].bias)
            return net
        self.actor, self.critic = mlp(ACT_DIM, 0.01), mlp(1, 1.0)

    def forward(self, x):
        return self.actor(x), self.critic(x).squeeze(-1)   # logits, normalised value


class RunningNorm:
    """Running mean/std of returns; the critic is trained on normalised returns."""
    def __init__(self):
        self.mean, self.var, self.count = 0.0, 1.0, 1e-4

    def update(self, x):
        bm, bv, bc = x.mean().item(), x.var(unbiased=False).item(), x.numel()
        d, tot = bm - self.mean, self.count + bc
        self.var = (self.var * self.count + bv * bc + d * d * self.count * bc / tot) / tot
        self.mean += d * bc / tot
        self.count = tot

    @property
    def std(self): return math.sqrt(max(self.var, 1e-8))
    def norm(self, x): return (x - self.mean) / self.std
    def denorm(self, x): return x * self.std + self.mean


def gae(rew, val, done, last_val):
    """rew/val/done: (T,E). Returns advantages and returns, both (T,E)."""
    T = rew.shape[0]
    adv = torch.zeros_like(rew)
    last, nxt = torch.zeros_like(last_val), last_val
    for t in reversed(range(T)):
        mask = 1.0 - done[t]
        delta = rew[t] + GAMMA * nxt * mask - val[t]
        last = delta + GAMMA * LAM * mask * last
        adv[t], nxt = last, val[t]
    return adv, adv + val


def _atomic_save(obj, path):
    torch.save(obj, path + ".tmp"); os.replace(path + ".tmp", path)


def export_npz(net, rn, path, info):
    """Weights in the NumPy format used by model.py (PPOAgent.load_npz) -> works with test_model.py."""
    a, c = net.actor, net.critic
    w = lambda l: l.weight.detach().cpu().numpy().T.astype(np.float64)
    b = lambda l: l.bias.detach().cpu().numpy().astype(np.float64)
    np.savez_compressed(
        path,
        a_W1=w(a[0]), a_b1=b(a[0]), a_W2=w(a[2]), a_b2=b(a[2]), a_W3=w(a[4]), a_b3=b(a[4]),
        c_W1=w(c[0]), c_b1=b(c[0]), c_W2=w(c[2]), c_b2=b(c[2]), c_W3=w(c[4]), c_b3=b(c[4]),
        episode=np.array(info["episodes"]), total_steps=np.array(info["env_steps"]),
        best_reward=np.array(info["best"]), entropy_coef=np.array(info["ent_coef"]),
        reward_history=np.array(info["hist"], dtype=np.float32),
        last_actor_loss=np.array(0.0), last_critic_loss=np.array(0.0),
        ret_norm_mean=np.array(rn.mean), ret_norm_var=np.array(rn.var), ret_norm_count=np.array(rn.count),
        max_speed=np.array(MAX_SPEED), max_steer_deg=np.array(MAX_STEER_DEG))


In [ ]:
# ═══════════════════════════ TRAIN ═══════════════════════════
def train():
    os.makedirs(OUT_DIR, exist_ok=True)
    torch.manual_seed(SEED); np.random.seed(SEED)
    E, T = NUM_ENVS, ROLLOUT
    net = ActorCritic().to(DEVICE)
    opt = torch.optim.Adam([{"params": net.actor.parameters(),  "lr": LR_ACTOR},
                            {"params": net.critic.parameters(), "lr": LR_CRITIC}], eps=1e-5)
    rn, ent_coef = RunningNorm(), ENT_COEF
    update, env_steps, episodes, best = 0, 0, 0, -float("inf")
    hist = collections.deque(maxlen=200)

    if RESUME:
        ck = torch.load(RESUME, map_location=DEVICE)
        net.load_state_dict(ck["net"]); opt.load_state_dict(ck["opt"])
        rn.mean, rn.var, rn.count = ck["rn"]
        ent_coef, update, env_steps = ck["ent_coef"], ck["update"], ck["env_steps"]
        # best score / episode history are NOT restored: they belong to the previous map,
        # and returns differ per map, so best.pt would otherwise never update on a new map.
        print(f"resumed from {RESUME} (update {update}, {env_steps:,} steps); best score reset")

    def snapshot(name, npz=False):
        info = dict(episodes=episodes, env_steps=env_steps, best=best, ent_coef=ent_coef, hist=list(hist))
        _atomic_save(dict(net=net.state_dict(), opt=opt.state_dict(), rn=(rn.mean, rn.var, rn.count),
                          ent_coef=ent_coef, update=update, env_steps=env_steps, episodes=episodes,
                          best=best, hist=list(hist)), os.path.join(OUT_DIR, name + ".pt"))
        if npz:
            export_npz(net, rn, os.path.join(OUT_DIR, name + ".npz"), info)

    cars = Cars(E)
    obs, _ = cars.observe()
    ep_ret = torch.zeros(E, device=DEVICE); ep_len = torch.zeros(E, device=DEVICE)
    recent_ret, recent_len, recent_crash = (collections.deque(maxlen=100) for _ in range(3))

    B_obs = torch.zeros(T, E, OBS_DIM, device=DEVICE)
    B_act = torch.zeros(T, E, dtype=torch.long, device=DEVICE)
    B_lp, B_val, B_rew, B_done = (torch.zeros(T, E, device=DEVICE) for _ in range(4))
    F_ret, F_len, F_crash = (torch.full((T, E), float("nan"), device=DEVICE) for _ in range(3))

    t0 = time.time(); t_log = t0; steps_log = env_steps
    print(f"training: {E} cars x {T} steps = {E*T:,} decisions/update, batch minibatch {MINIBATCH}")
    try:
        while env_steps < TOTAL_STEPS:
            if MAX_HOURS and time.time() - t0 > MAX_HOURS * 3600:
                print("time budget reached"); break

            # ── collect rollout ──
            F_ret.fill_(float("nan")); F_len.fill_(float("nan")); F_crash.fill_(float("nan"))
            with torch.no_grad():
                for t in range(T):
                    logits, raw = net(obs)
                    dist = Categorical(logits=logits)
                    act = dist.sample()
                    B_obs[t], B_act[t], B_lp[t], B_val[t] = obs, act, dist.log_prob(act), rn.denorm(raw)
                    obs, obs_final, rew, crashed, trunc = cars.step(act)
                    done = crashed | trunc
                    ep_ret += rew; ep_len += 1
                    F_ret[t] = torch.where(done, ep_ret, F_ret[t])
                    F_len[t] = torch.where(done, ep_len, F_len[t])
                    F_crash[t] = torch.where(done, crashed.float(), F_crash[t])
                    ep_ret = torch.where(done, torch.zeros_like(ep_ret), ep_ret)
                    ep_len = torch.where(done, torch.zeros_like(ep_len), ep_len)
                    if trunc.any():   # time-limit cut-off is not a failure: bootstrap its value
                        vf = rn.denorm(net.critic(obs_final).squeeze(-1))
                        rew = rew + GAMMA * vf * trunc.float()
                    B_rew[t], B_done[t] = rew, done.float()
                last_val = rn.denorm(net.critic(obs).squeeze(-1))
            env_steps += E * T

            fin = ~torch.isnan(F_ret)
            for r, l, c in zip(F_ret[fin].tolist(), F_len[fin].tolist(), F_crash[fin].tolist()):
                recent_ret.append(r); recent_len.append(l); recent_crash.append(c); hist.append(r)
                episodes += 1

            # ── PPO update ──
            adv, ret = gae(B_rew, B_val, B_done, last_val)
            rn.update(ret)
            ret_n = rn.norm(ret).reshape(-1)
            adv = ((adv - adv.mean()) / (adv.std() + 1e-8)).reshape(-1)
            f_obs, f_act, f_lp = B_obs.reshape(-1, OBS_DIM), B_act.reshape(-1), B_lp.reshape(-1)
            N = f_obs.shape[0]
            pg_l, v_l, ent_l, kl_l = [], [], [], []
            for _ in range(EPOCHS):
                perm = torch.randperm(N, device=DEVICE)
                for s in range(0, N, MINIBATCH):
                    i = perm[s:s + MINIBATCH]
                    logits, raw = net(f_obs[i])
                    dist = Categorical(logits=logits)
                    logr = dist.log_prob(f_act[i]) - f_lp[i]
                    ratio = logr.exp()
                    pg = -torch.min(ratio * adv[i], ratio.clamp(1 - CLIP_EPS, 1 + CLIP_EPS) * adv[i]).mean()
                    ent = dist.entropy().mean()
                    vl = 0.5 * ((raw - ret_n[i]) ** 2).mean()
                    opt.zero_grad(set_to_none=True)
                    (pg - ent_coef * ent + vl).backward()
                    nn.utils.clip_grad_norm_(net.parameters(), MAX_GRAD)
                    opt.step()
                    pg_l.append(pg.detach()); v_l.append(vl.detach()); ent_l.append(ent.detach())
                    kl_l.append(((ratio - 1) - logr).mean().detach())
                if torch.stack(kl_l[-(N // MINIBATCH or 1):]).mean() > 1.5 * TARGET_KL:
                    break
            ent_coef = max(ENT_MIN, ent_coef * ENT_DECAY)
            update += 1

            # ── log / save ──
            if update % LOG_EVERY == 0 or update == 1:
                now = time.time()
                mean_ret = float(np.mean(recent_ret)) if recent_ret else float("nan")
                mean_len = float(np.mean(recent_len)) if recent_len else float("nan")
                crash_pct = 100 * float(np.mean(recent_crash)) if recent_crash else float("nan")
                print(f"upd {update:>6} | steps {env_steps/1e6:7.2f}M | {(env_steps-steps_log)/(now-t_log):7.0f} st/s | "
                      f"eps {episodes:>7} | ret {mean_ret:8.1f} | surv {mean_len/FPS:5.1f}s | crash {crash_pct:3.0f}% | "
                      f"ent {torch.stack(ent_l).mean().item():.3f} | kl {torch.stack(kl_l).mean().item():.4f} | "
                      f"vloss {torch.stack(v_l).mean().item():.3f} | {(now-t0)/60:6.1f} min")
                t_log, steps_log = now, env_steps
                if len(recent_ret) >= 50 and mean_ret > best:
                    best = mean_ret
                    snapshot("best", npz=True)
                    print(f"           new best (mean return of last {len(recent_ret)} episodes): {best:.1f}")
            if update % SAVE_EVERY == 0:
                snapshot("latest")
            if update % SNAP_EVERY == 0:
                snapshot(f"snap_{update:05d}", npz=True)
    except KeyboardInterrupt:
        print("interrupted")
    finally:
        snapshot("final", npz=True)
        print(f"saved {OUT_DIR}/final.pt + final.npz  (best mean return {best:.1f}, {episodes} episodes, "
              f"{env_steps:,} decisions, {(time.time()-t0)/60:.1f} min)")
    return net


net = train()
